# Tutoriel 10

# Diffusivité variable et flux aux interfaces

Depuis le début du cours, `D` était une **constante**. À partir de cette semaine, `D` **dépend de la solution elle-même**, ce qui change trois choses dans la structure du code — trois sources d'erreur classiques.

Nous l'illustrons sur un **cône d'éboulis** : au pied d'une falaise, les blocs accumulés **fluent** sous leur propre poids et s'étalent sur des milliers d'années.

$$\frac{\partial h}{\partial t} = - \frac{\partial q_x}{\partial x}, \qquad q_x = -D(h) \frac{\partial h}{\partial x}, \qquad D(h) = k\, h^2$$

Toute la différence est dans la troisième expression : **plus le dépôt est épais, plus il flue vite**. C'est la structure de l'équation des glaciers de cette semaine, où $D$ dépend de l'épaisseur de glace à la puissance 5.

## 1. `D` n'a plus la taille de la solution

`D` sert à calculer un **flux**, c'est-à-dire ce qui passe d'une cellule à sa voisine : il ne vit pas *sur* un point de la grille mais *entre* deux points, et il est de taille `nx-1`. Mais `h` est de taille `nx` : **quelle épaisseur utiliser entre les points $i$ et $i+1$ ?** La **moyenne des deux**.

```
                               0     1    ...   i-1    i    i+1   ...  Taille
h                              |-----|-----|-----|-----|-----|----...    nx
                                  0     1    ...   i-1    i    i+1
hm   = 0.5*(h[1:]+h[:-1])         |-----|-----|-----|-----|-----|-...   nx-1
```

In [1]:
import numpy as np

Lx, k, nx = 100.0, 1e-3, 101              # domaine (m), coefficient de fluage (1/an)
x  = np.linspace(0, Lx, nx) ; dx = Lx/(nx-1)
h  = np.zeros(nx) ; h[np.abs(x-Lx/2) < 10] = 5.0    # un depot de 5 m au milieu

hm = 0.5*( h[1:] + h[:-1] )               # l'epaisseur ENTRE les points
D  = k * hm**2                            # la diffusivite, la ou vit le flux
qx = - D * ( h[1:] - h[:-1] )/dx          # le flux

print("h  :", h.shape,  " la solution, sur les noeuds")
print("hm :", hm.shape, " entre les noeuds : une cellule de moins")
print("D  :", D.shape,  " comme le flux")
print("qx :", qx.shape)
print()
print("-> la mise a jour s'ecrit :  h[1:-1] += - dt * ( qx[1:] - qx[:-1] )/dx")

h  : (101,)  la solution, sur les noeuds
hm : (100,)  entre les noeuds : une cellule de moins
D  : (100,)  comme le flux
qx : (100,)

-> la mise a jour s'ecrit :  h[1:-1] += - dt * ( qx[1:] - qx[:-1] )/dx


On pourrait écrire `D = k*h[:-1]**2` : la taille serait correcte et Python ne dirait rien. Mais ce choix **privilégie la cellule de gauche** dans un problème sans aucune asymétrie, et le dépôt se met à **dériver**. Le code tourne, le résultat *a l'air* plausible — d'où le réflexe : **si le problème est symétrique, vérifiez que la solution l'est aussi.**

## 2. Le pas de temps change à chaque itération

Puisque `D` dépend de `h` qui évolue, la contrainte de stabilité évolue aussi et doit être recalculée **dans** la boucle :

$$ dt_\mathrm{diff} = \frac{dx^2}{2.1 \times \max|D|}, \qquad dt = \min(dt_\mathrm{max},\, dt_\mathrm{diff}).$$

On prend le **maximum** de `D`, car l'endroit le plus « rapide » contraint tout le domaine. Et `D` vaut **zéro** là où il n'y a pas de dépôt, ce qui rendrait `dt_diff` infini : c'est ici que `dt_max` devient indispensable.

## 3. Le nombre d'itérations n'est plus connu à l'avance

Jusqu'ici on écrivait `nt = int(duree/dt)` avant la boucle. Ce n'est plus possible : `dt` change à chaque tour, donc le nombre d'itérations nécessaires est inconnu.

La solution : lancer la boucle sur un nombre volontairement très grand, et en **sortir** avec un `break` dès que le temps simulé est atteint.

In [2]:
dt_max, duree = 100.0, 20000.0     # pas de temps maximal (ans), duree a simuler (ans)
temps = 0.0

for it in range(100000):           # un nombre volontairement tres grand

    dt = min(dt_max, dx**2/(2.1*np.max(np.abs(D))))    # recalcule a chaque tour
    temps += dt

    if temps > duree:              # on sort des que la duree voulue est atteinte
        break

print("il a fallu", it, "iterations pour simuler", duree, "ans")

il a fallu 1050 iterations pour simuler 20000.0 ans


Une boucle `while temps < duree:` ferait exactement la même chose, en plus direct ; la version `for` + `break` a l'avantage de garantir une sortie au bout de 100 000 itérations même si `dt` devenait nul par erreur.

Notez qu'ici `D` n'est pas remis à jour, donc `dt` reste constant. Dans le modèle complet, `D` est recalculé à chaque tour à partir de `h`, `dt` augmente à mesure que le dépôt s'aplatit, et `dt_max` finit par prendre le relais.

## À expérimenter

1. Remplacez `hm = 0.5*(h[1:] + h[:-1])` par `hm = h[:-1]` dans la première cellule. Les tailles sont-elles toujours correctes ? Python signale-t-il quoi que ce soit ?
2. Doublez `nx`. Que devient `dt` ? Et le nombre d'itérations nécessaires ?